## WAV Channel Combiner with 100 Hz High-Pass Filter

Adjust the configuration below to select input/output folders and channel sets to combine. The processing loop will sum the specified microphone channels, apply a 100 Hz high-pass filter, and write the filtered result.

In [1]:
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np

try:
    import soundfile as sf
except Exception:
    sf = None  # type: ignore

try:
    from scipy.signal import butter, sosfiltfilt
except Exception:
    butter = None
    sosfiltfilt = None

try:
    from scipy.io import wavfile
except Exception:
    wavfile = None  # type: ignore


In [2]:
# Input folder containing multi-channel WAV files.
input_root = Path('path/to/input').expanduser()

# Output folder where the processed files will be stored.
output_root = Path('path/to/output').expanduser()

# High-pass filter settings.
high_pass_cutoff_hz = 100.0
butter_order = 4  # Used when scipy.signal is available

# Channel presets (1-based channel indices).
channel_presets: Dict[str, List[int]] = {
    'all': [3, 4, 5, 6],  # throat upper/lower + ear left/right
    'throat': [3, 4],
    'ears': [5, 6],
    'upper_throat': [3],
    'lower_throat': [4],
    'left_ear': [5],
    'right_ear': [6],
}

# Select which presets will be processed.
active_channel_sets: List[str] = ['all']

# Add custom combinations here if needed, for example:
# channel_presets['throat_left'] = [3, 5]
# active_channel_sets.append('throat_left')

# Search recursively for WAV files when True, otherwise only the top-level folder is scanned.
search_recursively = True


In [3]:
def load_multichannel_wav(filepath: Path) -> Tuple[np.ndarray, int]:
    """Return audio as float32 samples shaped (num_samples, num_channels)."""
    if sf is not None:
        data, sample_rate = sf.read(filepath, always_2d=True)
        if data.dtype != np.float32:
            data = data.astype(np.float32)
        return data, sample_rate

    if wavfile is None:
        raise RuntimeError('No audio loader available. Install soundfile or scipy.')

    sample_rate, data = wavfile.read(filepath)
    if data.ndim == 1:
        data = data[:, np.newaxis]

    original_dtype = data.dtype
    if np.issubdtype(original_dtype, np.signedinteger):
        info = np.iinfo(original_dtype)
        scale = max(abs(info.min), abs(info.max))
        if scale == 0:
            data = data.astype(np.float32)
        else:
            data = data.astype(np.float32) / float(scale)
    elif np.issubdtype(original_dtype, np.unsignedinteger):
        info = np.iinfo(original_dtype)
        data = data.astype(np.float32)
        center = (info.min + info.max) / 2.0
        span = (info.max - info.min) / 2.0
        if span == 0:
            data = data - center
        else:
            data = (data - center) / span
    else:
        data = data.astype(np.float32)

    return data, sample_rate


def write_audio(filepath: Path, audio: np.ndarray, sample_rate: int) -> None:
    filepath.parent.mkdir(parents=True, exist_ok=True)
    if sf is not None:
        sf.write(filepath, audio, sample_rate)
        return
    if wavfile is not None:
        clipped = np.clip(audio, -1.0, 1.0)
        wavfile.write(filepath, sample_rate, (clipped * np.iinfo(np.int16).max).astype(np.int16))
        return
    raise RuntimeError('No audio writer available. Install soundfile or scipy.')


def high_pass_filter(signal: np.ndarray, sample_rate: int, cutoff_hz: float, order: int = 4) -> np.ndarray:
    signal = signal.astype(np.float32)
    if butter is not None and sosfiltfilt is not None:
        sos = butter(order, cutoff_hz, btype='highpass', fs=sample_rate, output='sos')
        return sosfiltfilt(sos, signal).astype(np.float32)

    rc = 1.0 / (2.0 * np.pi * cutoff_hz)
    dt = 1.0 / float(sample_rate)
    alpha = rc / (rc + dt)
    filtered = np.empty_like(signal, dtype=np.float32)
    filtered[0] = signal[0]
    for index in range(1, signal.shape[0]):
        filtered[index] = alpha * (filtered[index - 1] + signal[index] - signal[index - 1])
    return filtered


def normalize_audio(audio: np.ndarray, target_peak: float = 0.99) -> np.ndarray:
    audio = audio.astype(np.float32)
    peak = float(np.max(np.abs(audio)))
    if peak == 0.0:
        return audio
    return (audio / peak * target_peak).astype(np.float32)


def process_wav_file(filepath: Path, selected_channels: List[int], channel_label: str) -> Path:
    audio, sample_rate = load_multichannel_wav(filepath)
    channel_indices = [channel - 1 for channel in selected_channels]
    invalid = [idx for idx in channel_indices if idx < 0 or idx >= audio.shape[1]]
    if invalid:
        raise ValueError(f'Channel indices {invalid} exceed available channels ({audio.shape[1]})')

    combined = np.sum(audio[:, channel_indices], axis=1)
    filtered = high_pass_filter(combined, sample_rate, high_pass_cutoff_hz, order=butter_order)
    normalized = normalize_audio(filtered)

    relative_path = filepath.relative_to(input_root)
    output_name = f"{filepath.stem}_{channel_label}_HPF.wav"
    output_path = (output_root / relative_path).with_name(output_name)

    write_audio(output_path, normalized, sample_rate)
    return output_path


In [4]:
if not input_root.exists():
    raise FileNotFoundError(f'Input folder not found: {input_root}')

output_root.mkdir(parents=True, exist_ok=True)

walker = input_root.rglob('*.wav') if search_recursively else input_root.glob('*.wav')
wav_files = sorted(walker)

if not wav_files:
    print(f'No WAV files found under {input_root}')
else:
    print(f'Found {len(wav_files)} wav files under {input_root}.')
    for preset_name in active_channel_sets:
        channels = channel_presets.get(preset_name)
        if channels is None:
            print(f"Skipping preset '{preset_name}' because it is not defined in channel_presets.")
            continue

        print(f"
Processing preset '{preset_name}' using channels {channels}.")
        for wav_path in wav_files:
            try:
                output_path = process_wav_file(wav_path, channels, preset_name)
                print(f"  [ok] {output_path.relative_to(output_root)}")
            except Exception as error:
                print(f"  [fail] {wav_path.name}: {error}")


SyntaxError: unterminated string literal (detected at line 19) (2404246776.py, line 19)